In [1]:
import sys
from pathlib import Path

# 将项目根目录添加到 Python 路径中，以便导入 src 模块
# notebook 位于 notebooks/ 目录下，项目根目录是上一级目录
notebook_dir = Path.cwd()  # 当前工作目录
# 如果当前目录是 notebooks，则项目根目录是父目录
# 否则尝试找到包含 src 目录的父目录
if (notebook_dir / "src").exists():
    project_root = notebook_dir
elif (notebook_dir.parent / "src").exists():
    project_root = notebook_dir.parent
else:
    # 默认假设 notebook 在 notebooks/ 目录下
    project_root = notebook_dir.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 切换到项目根目录，确保配置文件路径正确
import os
original_cwd = os.getcwd()
os.chdir(project_root)

from typing import List
import json
from src.agents.content_generate_agent.retrieval_workflow.llm_retriever import LLMRetriever
from src.agents.content_generate_agent.input_models import DocumentItems, Item
from loguru import logger
from src.agents.content_generate_agent.table_answer_components.table_retrieval.table_retrieval import retrieve_table_rag_batch, retrieve_table_llm_batch, retrieve_table_rag_single, retrieve_table_llm_single

# 使用绝对路径读取文件，避免工作目录变化的影响
content_file_path = project_root / "storage" / "20251111ccb2023plus.json"
with content_file_path.open("r", encoding="utf-8") as f:
    content_list = json.load(f)[0]["content"]

# 将原始内容先用强类型模型校验并转换为 Items
try:
    typed_items: List[Item] = DocumentItems.model_validate(content_list).root
except Exception as e:
    logger.error(f"内容列表校验失败：{e}")
    raise ValueError(f"内容列表校验失败，无法继续解析流程：{e}") from e

if len(typed_items) == 0:
    print("ERROR:内容列表为空，无法继续解析流程")
    raise ValueError("内容列表为空，无法继续解析流程")

question_tasks = [
    {
        "id": "#1",
        "text": "前五大下游客户情况",
        "prompt": "",
        "type": "question",
        "meta": {}
    },
    {
        "id": "#2",
        "text": "<table><tr><td><p></p></td><td><p>公司名称</p></td><td><p>是否关联企业</p></td><td><p>销售模式</p></td><td><p>是否签订长期合约</p></td><td><p>销售内容</p></td><td><p>销售额</p></td><td><p>销售占比</p></td><td><p>结算方式</p></td><td><p>付款周期</p></td><td><p>业务往来时间</p></td></tr><tr><td rowspan=\"5\"><p>前五大下游客户</p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td></tr><tr><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td></tr><tr><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td></tr><tr><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td></tr><tr><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td></tr><tr><td><p>合计</p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td><td><p></p></td></tr></table>",
        "prompt": "",
        "type": "table",
        "meta": {}
    }
]

questions = [q["text"] for q in question_tasks]
question_ids = [q["id"] for q in question_tasks]

print("--------------------------------")
print("RAG 批量检索结果:")
rag_results = await retrieve_table_rag_batch(questions, question_ids, typed_items, 5)
print([len(result["table_items"]) for result in rag_results])
print([result["table_items"] for result in rag_results])
print("--------------------------------")
rag_result = await retrieve_table_rag_single(questions[0], question_ids[0], typed_items, 5)
print(len(rag_result["table_items"]))
print(rag_result["table_items"])
print("--------------------------------")
print("LLM 批量检索结果:")
llm_results = await retrieve_table_llm_batch(questions, question_ids, typed_items)
print([len(result["table_items"]) for result in llm_results])
print([result["table_items"] for result in llm_results])
print("--------------------------------")


--------------------------------
RAG 批量检索结果:


2025-11-20 17:33:09.466 | INFO     | src.utils.langfuse_tracing:__init__:80 - Langfuse Tracer 初始化成功, Session ID: session-1763631189466
2025-11-20 17:33:09.519 | INFO     | src.agents.content_generate_agent.table_answer_components.table_retrieval.table_retrieval:retrieve_table_rag_batch:66 - 提取到 1 个唯一的file_id: ['95b4ab26682842f59f9159731ec42b6e7geNJ8pnRE']
2025-11-20 17:33:09.525 | DEBUG    | src.applications.rag.services.vectorstore:get_default_stores:682 - Milvus 集合解析: children=rag_grand_children, parents=rag_grand_children_parents, grandchildren=rag_grand_children_grandchildren
2025-11-20 17:33:10.065 | INFO     | src.applications.rag.services.query_service:run_batch_query:46 - 收到 RAG 查询请求，问题数量: 2, top_k: 5, return_child_chunks: False, rerank_enabled: False, hybrid_enabled: True, use_hybrid_pipeline: True, search_type: table
2025-11-20 17:33:10.066 | INFO     | src.applications.rag.services.query_service:run_batch_query:148 - 处理第 1/2 个问题: 前五大下游客户情况...
2025-11-20 17:33:10.068 | INFO  

[3, 5]
[[TableItem(page=15, type='table', bBox=BBox(l=49.0, t=717.0, r=534.0, b=769.0, coord_origin='TOPLEFT'), rows=[[TableCell(text='前五名客户合计销售金额（元）', rowspan=1, colspan=1, rowstart=0, colstart=0), TableCell(text='631,645,595.19', rowspan=1, colspan=1, rowstart=0, colstart=1)], [TableCell(text='前五名客户合计销售金额占年度销售总额比例', rowspan=1, colspan=1, rowstart=1, colstart=0), TableCell(text='81.66%', rowspan=1, colspan=1, rowstart=1, colstart=1)], [TableCell(text='前五名客户销售额中关联方销售额占年度销售总额比例', rowspan=1, colspan=1, rowstart=2, colstart=0), TableCell(text='0.00%', rowspan=1, colspan=1, rowstart=2, colstart=1)]], md='| 前五名客户合计销售金额（元） | 631,645,595.19 |\n| --- | --- |\n| 前五名客户合计销售金额占年度销售总额比例 | 81.66% |\n| 前五名客户销售额中关联方销售额占年度销售总额比例 | 0.00% |', html='<table><tr><td>前五名客户合计销售金额（元）</td><td>631,645,595.19</td></tr><tr><td>前五名客户合计销售金额占年度销售总额比例</td><td>81.66%</td></tr><tr><td>前五名客户销售额中关联方销售额占年度销售总额比例</td><td>0.00%</td></tr></table>', perfectTable=False, img_path='images/a82df5f3aef712cdf229cdc56d2a0bf583d95bb52

2025-11-20 17:33:12.016 | INFO     | src.applications.rag.adapters.embeddings.openai_embedder:embed_texts:72 - Batch 1/1 done, progress: 1/1
2025-11-20 17:33:12.016 | INFO     | src.applications.rag.adapters.embeddings.openai_embedder:embed_texts:73 - Embedding 完成total_embedded=1
2025-11-20 17:33:12.018 | DEBUG    | src.applications.rag.services.vectorstore:get_relevant_documents:348 - 应用过滤: (metadata["file_id"] == "95b4ab26682842f59f9159731ec42b6e7geNJ8pnRE") && (metadata["type"] == "table")
2025-11-20 17:33:12.262 | DEBUG    | src.applications.rag.services.vectorstore:get_relevant_documents:360 - 向量检索返回 42 条原始结果（已应用file_id过滤）
2025-11-20 17:33:12.264 | INFO     | src.applications.rag.services.query_service:run_batch_query:183 - 问题 1 检索完成：向量候选 42 条，BM25 候选 0 条，执行 RRF 融合
2025-11-20 17:33:12.265 | INFO     | src.applications.rag.services.query_service:run_batch_query:188 - 问题 1 融合后候选 42 条
2025-11-20 17:33:12.266 | INFO     | src.applications.rag.services.query_service:run_batch_query:349

3
[TableItem(page=15, type='table', bBox=BBox(l=49.0, t=717.0, r=534.0, b=769.0, coord_origin='TOPLEFT'), rows=[[TableCell(text='前五名客户合计销售金额（元）', rowspan=1, colspan=1, rowstart=0, colstart=0), TableCell(text='631,645,595.19', rowspan=1, colspan=1, rowstart=0, colstart=1)], [TableCell(text='前五名客户合计销售金额占年度销售总额比例', rowspan=1, colspan=1, rowstart=1, colstart=0), TableCell(text='81.66%', rowspan=1, colspan=1, rowstart=1, colstart=1)], [TableCell(text='前五名客户销售额中关联方销售额占年度销售总额比例', rowspan=1, colspan=1, rowstart=2, colstart=0), TableCell(text='0.00%', rowspan=1, colspan=1, rowstart=2, colstart=1)]], md='| 前五名客户合计销售金额（元） | 631,645,595.19 |\n| --- | --- |\n| 前五名客户合计销售金额占年度销售总额比例 | 81.66% |\n| 前五名客户销售额中关联方销售额占年度销售总额比例 | 0.00% |', html='<table><tr><td>前五名客户合计销售金额（元）</td><td>631,645,595.19</td></tr><tr><td>前五名客户合计销售金额占年度销售总额比例</td><td>81.66%</td></tr><tr><td>前五名客户销售额中关联方销售额占年度销售总额比例</td><td>0.00%</td></tr></table>', perfectTable=False, img_path='images/a82df5f3aef712cdf229cdc56d2a0bf583d95bb5228fb0b

2025-11-20 17:33:12.897 | INFO     | src.agents.content_generate_agent.retrieval_workflow.block_manager:arrange_blocks:206 - 处理 block 0，包含 45 个 texts
2025-11-20 17:33:12.914 | INFO     | src.agents.content_generate_agent.retrieval_workflow.block_manager:arrange_blocks:297 -   -> 生成 20 个 pieces
2025-11-20 17:33:12.915 | INFO     | src.agents.content_generate_agent.retrieval_workflow.block_manager:arrange_blocks:206 - 处理 block 1，包含 23 个 texts
2025-11-20 17:33:12.930 | INFO     | src.agents.content_generate_agent.retrieval_workflow.block_manager:arrange_blocks:297 -   -> 生成 15 个 pieces
2025-11-20 17:33:12.931 | INFO     | src.agents.content_generate_agent.retrieval_workflow.block_manager:arrange_blocks:206 - 处理 block 2，包含 29 个 texts
2025-11-20 17:33:12.953 | INFO     | src.agents.content_generate_agent.retrieval_workflow.block_manager:arrange_blocks:297 -   -> 生成 17 个 pieces
2025-11-20 17:33:12.956 | INFO     | src.agents.content_generate_agent.retrieval_workflow.block_manager:arrange_blo

[26, 43]
[[TableItem(page=15, type='table', bBox=BBox(l=49.0, t=193.0, r=534.0, b=260.0, coord_origin='TOPLEFT'), rows=[[TableCell(text='行业分类', rowspan=1, colspan=1, rowstart=0, colstart=0), TableCell(text='项目', rowspan=1, colspan=1, rowstart=0, colstart=1), TableCell(text='单位', rowspan=1, colspan=1, rowstart=0, colstart=2), TableCell(text='2023年', rowspan=1, colspan=1, rowstart=0, colstart=3), TableCell(text='2022年', rowspan=1, colspan=1, rowstart=0, colstart=4), TableCell(text='同比增减', rowspan=1, colspan=1, rowstart=0, colstart=5)], [TableCell(text='饮料包装行业', rowspan=3, colspan=1, rowstart=1, colstart=0), TableCell(text='销售量', rowspan=1, colspan=1, rowstart=1, colstart=1), TableCell(text='万个', rowspan=1, colspan=1, rowstart=1, colstart=2), TableCell(text='1,901,583.24', rowspan=1, colspan=1, rowstart=1, colstart=3), TableCell(text='1,837,720.82', rowspan=1, colspan=1, rowstart=1, colstart=4), TableCell(text='3.48%', rowspan=1, colspan=1, rowstart=1, colstart=5)], [TableCell(text='生产量',